# Binary Search and Approximations

A single idea powers this entire chapter:

> If you know a function is **monotone** (always increasing or always decreasing),
> you can find any target value in it by repeatedly halving your search interval.

You will apply this idea to approximate square roots, cube roots, nth roots,
and logarithms — without ever using `math.sqrt`, `math.log`, or `**`.

In [ ]:
import math
import matplotlib.pyplot as plt

---
## Part 0 — The Core Idea

### Exercise 0.1 — Guess My Number

Suppose someone thinks of an integer between 1 and 1000 (inclusive).
You can ask "Is it less than, equal to, or greater than X?" and they will answer honestly.

**Question 1:** If you always guess randomly, how many guesses might you need in the worst case?

**Question 2:** If you always guess the *middle* of the remaining range, how many guesses
do you need in the worst case? Try it on paper for the range [1, 1000].

**Key insight:** Each "guess the middle" halves the range. After k guesses the range has
shrunk from 1000 to roughly 1000 / 2^k.
How many times can you halve 1000 before you reach 1?

```python
# How many halvings to go from n to 1?
# Each step: n -> n/2 -> n/4 -> ...
# After k steps: n / 2^k <= 1  →  k >= log2(n)
```

**Write a function** `count_halvings(n)` that returns how many times you can halve `n`
(starting from a whole number) before reaching 1 or below.

```python
count_halvings(1000)  # about 10
count_halvings(1)     # 0
count_halvings(16)    # 4
```

In [ ]:
def count_halvings(n):
    pass  # replace this


assert count_halvings(1) == 0
assert count_halvings(16) == 4
assert count_halvings(1000) == 9   # floor(log2(1000))
print("count_halvings: OK")
print(f"To find a number in [1..1000], at most {count_halvings(1000)+1} guesses needed.")

---
## Part 1 — Square Root via Binary Search

The square root of `n` is the number `x` such that `x * x = n`.
For example, sqrt(25) = 5 because 5 * 5 = 25.

**Before you code — think about the search:**

1. If you want sqrt(25), you know the answer is between **0** and **25**. Why?
2. Guess `mid = (0 + 25) / 2 = 12.5`. Compute `12.5 * 12.5 = 156.25`.
   Is that too big or too small compared to 25?
3. So the answer must be in the range [0, 12.5]. Next mid = 6.25.
   Compute `6.25 * 6.25 = 39.06`. Still too big. Range becomes [0, 6.25].
4. Continue until `mid * mid` is within `precision` of `n`.

**Stopping rule:** Stop when `abs(mid * mid - n) < precision`.

### Exercise 1.1 — Square Root

Write `sqrt_binary_search(n, precision)` that returns an approximation of sqrt(n).

**Arguments:**
- `n`: a non-negative number
- `precision`: how close `mid*mid` must be to `n` (e.g., 0.0001)

**Before you code:**
- What is the initial `low`? What is the initial `high`?
- When `mid*mid < n`, should you move `low` or `high` to `mid`?
- When does the loop stop?

```python
abs(sqrt_binary_search(25,  0.0001) - 5.0) < 0.001    # True
abs(sqrt_binary_search(2,   0.0001) - 1.4142) < 0.001 # True
abs(sqrt_binary_search(0,   0.0001) - 0.0) < 0.001    # True
abs(sqrt_binary_search(144, 0.0001) - 12.0) < 0.001   # True
```

In [ ]:
def sqrt_binary_search(n, precision):
    pass  # replace this


assert abs(sqrt_binary_search(25,  0.0001) - 5.0)    < 0.001
assert abs(sqrt_binary_search(2,   0.0001) - 1.4142) < 0.001
assert abs(sqrt_binary_search(0,   0.0001) - 0.0)    < 0.001
assert abs(sqrt_binary_search(144, 0.0001) - 12.0)   < 0.001
print("sqrt_binary_search: OK")
print(f"sqrt(2) ≈ {sqrt_binary_search(2, 1e-9):.9f}  (math.sqrt = {math.sqrt(2):.9f})")

#### Plotting helper — run as-is

This shows how the search interval [low, high] narrows with each step.

In [ ]:
def plot_binary_search_convergence(n, precision, title="Binary search convergence"):
    low, high = 0, max(1, n)
    intervals = [(low, high)]
    while True:
        mid = (low + high) / 2
        if abs(mid * mid - n) < precision:
            break
        if mid * mid < n:
            low = mid
        else:
            high = mid
        intervals.append((low, high))

    steps = range(len(intervals))
    widths = [h - l for l, h in intervals]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(steps, widths, marker="o", ms=3)
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("interval width")
    axes[0].set_title("Interval narrows each step")
    axes[0].set_yscale("log")
    mids = [(l + h) / 2 for l, h in intervals]
    axes[1].plot(steps, mids, marker="o", ms=3)
    axes[1].axhline(math.sqrt(n), color="red", linestyle="--", label=f"true sqrt({n})")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("midpoint guess")
    axes[1].set_title("Midpoint converges to answer")
    axes[1].legend()
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    print(f"Converged in {len(intervals)} steps.")

plot_binary_search_convergence(2, 1e-9, title="sqrt(2) search")

---
## Part 2 — Cube Root via Binary Search

The cube root of `n` is the number `x` such that `x * x * x = n`.
For example, cuberoot(27) = 3 because 3 * 3 * 3 = 27.

This is more interesting than sqrt because:
- The cube root of a **negative** number is also negative (cuberoot(-8) = -2).
- For `|n| < 1`, the cube root is **larger** than `|n|` (cuberoot(0.001) = 0.1).

**Before you code — think about these cases:**

1. cuberoot(27):  search range [0, 27]. Standard.
2. cuberoot(-64): cube root of |-64| = 4, then negate → -4.
3. cuberoot(0.001): |n| < 1, so the answer is between 0 and 1, not 0 and 0.001.
   Use range [0, 1] when |n| < 1.

**Stopping rule:** Stop when `abs(mid**3 - abs(n)) < precision`.

### Exercise 2.1 — Cube Root

Write `cuberoot_binary_search(n, precision)`.

**Steps:**
1. Handle `n == 0` (return 0).
2. Work on `abs_n = abs(n)`.
3. Set `low = 0`, `high = abs_n` if `abs_n >= 1`, else `high = 1`.
4. Binary search until `abs(mid**3 - abs_n) < precision`.
5. If `n < 0`, return `-mid`.

```python
abs(cuberoot_binary_search(27,    1e-6) - 3.0)   < 1e-4  # True
abs(cuberoot_binary_search(8,     1e-6) - 2.0)   < 1e-4  # True
abs(cuberoot_binary_search(-64,   1e-6) - (-4.0)) < 1e-4  # True
abs(cuberoot_binary_search(0.001, 1e-6) - 0.1)   < 1e-4  # True
abs(cuberoot_binary_search(0,     1e-6) - 0.0)   < 1e-4  # True
```

In [ ]:
def cuberoot_binary_search(n, precision):
    pass  # replace this


assert abs(cuberoot_binary_search(27,    1e-6) - 3.0)    < 1e-4
assert abs(cuberoot_binary_search(8,     1e-6) - 2.0)    < 1e-4
assert abs(cuberoot_binary_search(-64,   1e-6) - (-4.0)) < 1e-4
assert abs(cuberoot_binary_search(0.001, 1e-6) - 0.1)    < 1e-4
assert abs(cuberoot_binary_search(0,     1e-6) - 0.0)    < 1e-4
print("cuberoot_binary_search: OK")
print(f"cuberoot(-125) ≈ {cuberoot_binary_search(-125, 1e-9):.6f}  (expect -5.0)")

---
## Part 3 — Nth Root via Binary Search

You just wrote `sqrt_binary_search` and `cuberoot_binary_search`.
Look at them side by side. The only thing that changes is the **check function**:

| Function | Check |
|---|---|
| sqrt | `mid * mid` |
| cuberoot | `mid * mid * mid` |
| nth root | `mid ** n` |

**Before you code:**
- What range should `low` and `high` be for the nth root?
  - If `x >= 1`: range is [0, x] (the nth root is always ≤ x for n ≥ 1).
  - If `0 < x < 1`: the nth root is **larger** than x, so use range [0, 1].
- For **odd** n, negative x values have real roots. For **even** n, they do not.

### Exercise 3.1 — Nth Root

Write `nth_root_binary_search(x, n, precision=1e-9)` that returns the real nth root of x.

For simplicity, assume x >= 0 (handle negatives as a bonus below).

```python
abs(nth_root_binary_search(16,  2) - 4.0)  < 1e-6   # sqrt(16)=4
abs(nth_root_binary_search(27,  3) - 3.0)  < 1e-6   # cuberoot(27)=3
abs(nth_root_binary_search(81,  4) - 3.0)  < 1e-6   # 4th root of 81=3
abs(nth_root_binary_search(32,  5) - 2.0)  < 1e-6   # 5th root of 32=2
abs(nth_root_binary_search(1,   7) - 1.0)  < 1e-6   # any root of 1 is 1
abs(nth_root_binary_search(0.5, 2) - math.sqrt(0.5)) < 1e-6
```

In [ ]:
def nth_root_binary_search(x, n, precision=1e-9):
    pass  # replace this


assert abs(nth_root_binary_search(16,  2) - 4.0)             < 1e-6
assert abs(nth_root_binary_search(27,  3) - 3.0)             < 1e-6
assert abs(nth_root_binary_search(81,  4) - 3.0)             < 1e-6
assert abs(nth_root_binary_search(32,  5) - 2.0)             < 1e-6
assert abs(nth_root_binary_search(1,   7) - 1.0)             < 1e-6
assert abs(nth_root_binary_search(0.5, 2) - math.sqrt(0.5))  < 1e-6
print("nth_root_binary_search: OK")

### Bonus 3.2 — Verify with Python's built-in

Python can compute nth roots with `x ** (1/n)`.
Verify that your function gives the same answer for a range of values.

In [ ]:
for x, n in [(16, 2), (27, 3), (81, 4), (32, 5), (2, 10)]:
    mine = nth_root_binary_search(x, n)
    builtin = x ** (1 / n)
    print(f"nth_root_binary_search({x}, {n}) = {mine:.9f}  |  {x}**(1/{n}) = {builtin:.9f}  |  diff = {abs(mine-builtin):.2e}")

---
## Part 4 — Log Base 10 via Binary Search

So far you searched for a VALUE `x` such that `x**n == target`.
Now flip the problem: search for an **EXPONENT** `y` such that `10**y == x`.

This is exactly the definition of the logarithm:

$$\log_{10}(x) = y \quad \Leftrightarrow \quad 10^y = x$$

**Why does binary search still work?**
The function `f(y) = 10**y` is strictly increasing.
So if `10**mid < x`, the true exponent must be larger: move `low = mid`.
If `10**mid > x`, move `high = mid`.

**Bracket the search range:**
- If `x >= 1`: we need `y >= 0`. Start with `low = 0, high = 1`.
  Keep doubling `high` until `10**high >= x`.
- If `0 < x < 1`: we need `y < 0`. Start with `low = -1, high = 0`.
  Keep doubling the magnitude of `low` (i.e., `low *= 2`) until `10**low <= x`.

**Stopping rule:** `high - low < 1e-12` or `abs(10**mid - x) <= tol * max(1, x)`.

### Exercise 4.1 — Log Base 10

Write `log10_binary_search(x, tol=1e-7)`.

**Steps:**
1. Raise `ValueError` if `x <= 0`.
2. Return `0.0` if `x == 1`.
3. Find the bracket `[low, high]`:
   - If `x >= 1`: start `low=0, high=1`; keep `high *= 2` until `10**high >= x`.
   - If `x < 1`: start `low=-1, high=0`; keep `low *= 2` until `10**low <= x`.
4. Binary search: `mid = (low+high)/2`; adjust `low` or `high`.
5. Stop when `high - low < 1e-12`.
6. Return `(low + high) / 2`.

```python
abs(log10_binary_search(100)  - 2.0) < 1e-6   # True
abs(log10_binary_search(1000) - 3.0) < 1e-6   # True
abs(log10_binary_search(0.01) - (-2.0)) < 1e-6 # True
abs(log10_binary_search(1)    - 0.0) < 1e-9   # True
abs(round(log10_binary_search(2), 5) - 0.30103) < 1e-5  # True
```

In [ ]:
def log10_binary_search(x, tol=1e-7):
    pass  # replace this


assert abs(log10_binary_search(100)  - 2.0)    < 1e-6
assert abs(log10_binary_search(1000) - 3.0)    < 1e-6
assert abs(log10_binary_search(0.01) - (-2.0)) < 1e-6
assert abs(log10_binary_search(1)    - 0.0)    < 1e-9
assert abs(log10_binary_search(2)    - math.log10(2)) < 1e-6
print("log10_binary_search: OK")
print(f"log10(2) ≈ {log10_binary_search(2):.8f}  (math.log10 = {math.log10(2):.8f})")

#### Plotting helper — run as-is

Visualise the search for the exponent y such that 10^y = x.

In [ ]:
def plot_log10_search(x):
    if x <= 0:
        raise ValueError("x must be positive")
    if x >= 1:
        low, high = 0.0, 1.0
        while 10**high < x:
            high *= 2
    else:
        low, high = -1.0, 0.0
        while 10**low > x:
            low *= 2
    mids = []
    for _ in range(60):
        mid = (low + high) / 2
        mids.append(mid)
        if high - low < 1e-12:
            break
        if 10**mid < x:
            low = mid
        else:
            high = mid
    true_log = math.log10(x)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(range(len(mids)), mids, marker="o", ms=3, label="mid guess for y")
    ax.axhline(true_log, color="red", linestyle="--", label=f"true log10({x}) = {true_log:.5f}")
    ax.set_xlabel("step")
    ax.set_ylabel("y (exponent guess)")
    ax.set_title(f"Binary search for log10({x})")
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_log10_search(2)

---
## Part 5 — Log Base N via Binary Search

Now generalise: instead of searching for `y` such that `10**y == x`,
search for `y` such that `n**y == x` for any base `n`.

**When n > 1:** `f(y) = n**y` is strictly increasing. Same logic as log10.
**When 0 < n < 1:** `f(y) = n**y` is strictly **decreasing**.
For example: `0.5**2 = 0.25 < 0.5**1 = 0.5`. As y increases, n^y decreases.

For the case 0 < n < 1 and the comparisons flip:
- If `n**mid > x`, we need a larger exponent → `low = mid`.
- If `n**mid < x`, we need a smaller exponent → `high = mid`.

**Before you code:**
- Verify by hand: `log_base_n(0.04, 5) = -2` because `5**(-2) = 1/25 = 0.04`.
- With base=5 (> 1), `f(y) = 5**y` is increasing. x=0.04 < 1 so y must be negative.
  Start with `low=-1, high=0`; keep `low *= 2` until `5**low <= 0.04`.

**Raise `ValueError`** if `x <= 0`, `n <= 0`, or `n == 1`.

### Exercise 5.1 — Log Base N

Write `log_base_n(x, n, tol=1e-7)`.

The bracket-finding logic is the same as log10, but with `n**y` instead of `10**y`.
For `0 < n < 1`, you will need to flip the comparison in the binary search step.

```python
abs(log_base_n(8,    2) - 3.0)   < 1e-6   # 2^3 = 8
abs(log_base_n(81,   3) - 4.0)   < 1e-6   # 3^4 = 81
abs(log_base_n(0.04, 5) - (-2.0)) < 1e-6  # 5^(-2) = 0.04
abs(log_base_n(10,   2) - math.log(10, 2)) < 1e-6
abs(log_base_n(100, 10) - 2.0)   < 1e-6   # same as log10
```

In [ ]:
def log_base_n(x, n, tol=1e-7):
    pass  # replace this


assert abs(log_base_n(8,    2) - 3.0)             < 1e-6
assert abs(log_base_n(81,   3) - 4.0)             < 1e-6
assert abs(log_base_n(0.04, 5) - (-2.0))          < 1e-6
assert abs(log_base_n(10,   2) - math.log(10, 2)) < 1e-6
assert abs(log_base_n(100, 10) - 2.0)             < 1e-6
print("log_base_n: OK")
print(f"log2(10) ≈ {log_base_n(10, 2):.8f}  (math.log = {math.log(10, 2):.8f})")

### Bonus 5.2 — Change of Base Formula

The **change of base formula** says:

$$\log_b(x) = \frac{\log_a(x)}{\log_a(b)}$$

Verify this using your `log_base_n` function:
`log_base_n(x, b)` should equal `log_base_n(x, a) / log_base_n(b, a)` for any a.

```python
# log2(8) using base-10 logs:
# log10(8) / log10(2) = log2(8)
```

In [ ]:
def verify_change_of_base(x, b, a=10):
    direct = log_base_n(x, b)
    via_change = log_base_n(x, a) / log_base_n(b, a)
    print(f"log_{b}({x}) directly    = {direct:.8f}")
    print(f"log_{b}({x}) via base {a} = {via_change:.8f}")
    print(f"Difference: {abs(direct - via_change):.2e}")
    assert abs(direct - via_change) < 1e-5
    print("Change of base verified!")

verify_change_of_base(8, 2)
verify_change_of_base(81, 3)
verify_change_of_base(1000, 10)

---
## Summary

You applied **one idea** — binary search on a continuous range — to five different problems:

| Function | Searching for | Check condition |
|---|---|---|
| `sqrt_binary_search(n, p)` | x such that x² = n | `mid*mid` vs `n` |
| `cuberoot_binary_search(n, p)` | x such that x³ = n | `mid**3` vs `n` (handle sign) |
| `nth_root_binary_search(x, n, p)` | x such that x^n = target | `mid**n` vs `target` |
| `log10_binary_search(x, tol)` | y such that 10^y = x | `10**mid` vs `x` |
| `log_base_n(x, n, tol)` | y such that n^y = x | `n**mid` vs `x` (flip if n<1) |

**Binary search works whenever the function is monotone** — that single property
guarantees the interval always shrinks and the answer is always inside it.